# Curating LibriSpeech with NeMo Curator

This notebook reads a prepared English LibriSpeech manifest, runs NeMo ASR, computes WER and duration, filters rows, and writes JSONL output. Dataset preparation is separate from pipeline execution, so the pipeline begins with `ManifestReader`.

## 1. Prepare a small real-data input

Run this once from the repository root before executing the remaining cells:

```bash
python benchmarking/data_prep/prepare_librispeech_data.py \
  --output-path ./example_audio/librispeech \
  --target-audio-hours 0.25
```

The pinned `openslr/librispeech_asr` source is English and CC BY 4.0 licensed. The command produces `manifest.jsonl` and an `audio/` directory containing FLAC files.

In [ ]:
import json
import shutil
from pathlib import Path

from nemo_curator.backends.xenna import XennaExecutor
from nemo_curator.core.client import RayClient
from nemo_curator.pipeline import Pipeline
from nemo_curator.stages.audio import ManifestReader
from nemo_curator.stages.audio.common import GetAudioDurationStage, PreserveByValueStage
from nemo_curator.stages.audio.inference.asr.stage import ASRStage
from nemo_curator.stages.audio.io.convert import AudioToDocumentStage
from nemo_curator.stages.audio.metrics.wer import GetPairwiseWerStage
from nemo_curator.stages.resources import Resources
from nemo_curator.stages.text.io.writer import JsonlWriter

MANIFEST_PATH = Path("./example_audio/librispeech/manifest.jsonl").resolve()
OUTPUT_DIR = Path("./example_audio/librispeech/result-notebook").resolve()
MODEL_ID = "nvidia/parakeet-tdt-0.6b-v2"
WER_THRESHOLD = 1000.0

if not MANIFEST_PATH.is_file():
    message = f"Prepare the manifest first: {MANIFEST_PATH}"
    raise FileNotFoundError(message)

In [ ]:
def build_pipeline() -> Pipeline:
    pipeline = Pipeline(name="librispeech_tutorial", description="ASR and WER curation for prepared LibriSpeech")
    pipeline.add_stage(ManifestReader(manifest_path=str(MANIFEST_PATH)))
    pipeline.add_stage(
        ASRStage(
            adapter_target="nemo_curator.models.asr.nemo_asr.NeMoASRAdapter",
            model_id=MODEL_ID,
            audio_filepath_key="audio_filepath",
            target_sample_rate=16000,
            batch_size=16,
            adapter_kwargs={"use_cuda_graph_decoder": False},
        ).with_(resources=Resources(gpus=1.0))
    )
    pipeline.add_stage(GetPairwiseWerStage(text_key="text", pred_text_key="pred_text", wer_key="wer_pct"))
    pipeline.add_stage(GetAudioDurationStage(audio_filepath_key="audio_filepath", duration_key="duration"))
    pipeline.add_stage(PreserveByValueStage(input_value_key="wer_pct", target_value=WER_THRESHOLD, operator="le"))
    pipeline.add_stage(AudioToDocumentStage())
    pipeline.add_stage(JsonlWriter(path=str(OUTPUT_DIR), write_kwargs={"force_ascii": False}))
    return pipeline

print(build_pipeline().describe())

In [ ]:
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
ray_client = RayClient()
try:
    ray_client.start()
    build_pipeline().run(XennaExecutor())
finally:
    ray_client.stop()

In [ ]:
rows = []
for output_file in sorted(OUTPUT_DIR.rglob("*.jsonl")):
    with output_file.open(encoding="utf-8") as stream:
        rows.extend(json.loads(line) for line in stream if line.strip())

print(f"Rows retained: {len(rows)}")
if rows:
    print(json.dumps(rows[0], indent=2, ensure_ascii=False))
    wers = [row["wer_pct"] for row in rows]
    print(f"WER range: {min(wers):.2f}% to {max(wers):.2f}%")

## Choose a curation threshold

The default is deliberately permissive. LibriSpeech references are uppercase and unpunctuated, while the model may emit different case and punctuation; `GetPairwiseWerStage` compares the supplied strings directly. Inspect the distribution, then lower `WER_THRESHOLD` to match your normalization and quality policy.